In [10]:
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms, datasets
from torchinfo import summary
from torch.utils.data import DataLoader, Subset

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

In [3]:
device

'mps'

In [4]:
IMG_SIZE = 224
seed = 42

In [5]:
def to_rgb(img):
    return img.convert("RGB")

In [8]:
train_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.RandomResizedCrop(IMG_SIZE, scale = (0.6,1), ratio = (0.75,1.33)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue = 0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), #imagenet standard values
    #transforms.RandomErasing(p=0.25, scale = (0.02, 0.15), ratio =(0.3, 3.3), value = "random")
])

In [9]:
test_tf = transforms.Compose([
    transforms.Lambda(to_rgb),
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [11]:
from torch.utils.data import random_split
root = "../data"
test_ratio = 0.2



In [12]:
ds_train_full = datasets.Caltech101(root =root, download = True, transform = train_tf) 
ds_test_full = datasets.Caltech101(root =root, download = True, transform = test_tf) 

100%|████████████████████████████████████████| 137M/137M [00:08<00:00, 16.1MB/s]


In [13]:
n_total = len(ds_train_full)

In [14]:
n_total

8677

In [15]:
n_test = int(n_total * test_ratio)

In [16]:
n_total, n_test

(8677, 1735)

In [17]:
n_train = n_total - n_test

In [18]:
n_total, n_test, n_train

(8677, 1735, 6942)

In [19]:
g = torch.Generator().manual_seed(seed)
perm = torch.randperm(n_total,generator = g).tolist()

In [21]:
train_idx = perm[:n_train]
test_idx = perm[n_train:]

In [22]:
len(train_idx), len(test_idx)

(6942, 1735)

In [23]:
train_ds = Subset(ds_train_full, train_idx)
test_ds = Subset(ds_test_full, test_idx)

In [24]:
len(train_ds), len(test_ds)

(6942, 1735)

In [25]:
BATCH_SIZE = 32
train_dataloader = DataLoader(train_ds, batch_size = BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_ds, batch_size = BATCH_SIZE, shuffle=False)

In [31]:
img, label = ds_train_full[1000]

In [32]:
print(label)
print(ds_train_full.categories[label])

2
Leopards
